# Diagnóstico do filtro espectral do C-NBI

Este notebook reconstrói e audita todas as combinações candidatas de cardinalidade $k=2,3,4$ consideradas pelo C-NBI nas dez sementes da campanha FULL. Cada combinação é classificada em uma das quatro categorias mutuamente exclusivas:

1. **aprovada**: $\sigma_{\min}$ supera o piso espectral e $q_C$ não supera o teto de condicionamento;
2. **rejeitada por $\sigma_{\min}$**: falha apenas no piso espectral;
3. **rejeitada por $q_C$**: falha apenas no teto de condicionamento;
4. **rejeitada por ambos**: falha simultaneamente nos dois critérios.

Os limiares são lidos diretamente da análise paralela da campanha FULL. O payoff normalizado é reconstruído com a mesma matriz de planejamento, ruído, RSM, pontos iniciais e otimizador usados no pipeline. As combinações reconstruídas como aprovadas são conferidas contra os checkpoints CNBI existentes.

In [ ]:
from pathlib import Path
from itertools import combinations, product
from math import comb
import json
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap

def project_root(start=Path.cwd()):
    path = start.resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'configs' / 'full.json').exists():
            return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada.')

ROOT = project_root()
SCENARIO_DIR = ROOT / 'data' / 'generated'
PARALLEL_PATH = ROOT / 'results' / 'synthetic' / 'tables' / 'full_parallel_analysis.csv'
RUNS_PATH = ROOT / 'results' / 'synthetic' / 'tables' / 'full_deterministic_runs.csv'
OUT_DIR = ROOT / 'results' / 'synthetic' / 'figures' / 'spectral_filtering'
OUT_DIR.mkdir(parents=True, exist_ok=True)

M_VALUES = (4, 6, 12)
CORRELATION_LEVELS = ('low', 'medium', 'high')
SCENARIOS = [f'm{m}_{level}' for m in M_VALUES for level in CORRELATION_LEVELS]
K_VALUES = (2, 3, 4)
ALPHA = 2 ** 0.75
FIGURE_WIDTH_CM = 16.0
CM_TO_INCH = 1 / 2.54

CATEGORY_ORDER = [
    'approved',
    'rejected_sigma_min',
    'rejected_qc',
    'rejected_both',
]
CATEGORY_LABELS = {
    'approved': 'aprovadas',
    'rejected_sigma_min': r'rejeitadas por $\sigma_{\min}$',
    'rejected_qc': r'rejeitadas por $q_C$',
    'rejected_both': 'rejeitadas por ambos',
}
CATEGORY_COLORS = {
    'approved': '#2166AC',
    'rejected_sigma_min': '#F4A582',
    'rejected_qc': '#D6604D',
    'rejected_both': '#B2182B',
}
CATEGORY_TEXT_COLORS = {
    'approved': 'white',
    'rejected_sigma_min': '0.12',
    'rejected_qc': 'white',
    'rejected_both': 'white',
}
RDBU_NO_WHITE = LinearSegmentedColormap.from_list(
    'RdBu_no_white',
    [(0.00, '#B2182B'), (0.499, '#F4A582'),
     (0.501, '#92C5DE'), (1.00, '#2166AC')],
)

plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 9.5,
    'axes.titlesize': 10.5,
    'axes.labelsize': 9.5,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 8.0,
    'legend.fontsize': 8.5,
    'savefig.facecolor': 'white',
    'axes.facecolor': 'white',
})

print('Saída:', OUT_DIR)

In [ ]:
def z(x):
    x1, x2, x3 = np.asarray(x, dtype=float)
    return np.array([1, x1, x2, x3, x1*x1, x2*x2, x3*x3, x1*x2, x1*x3, x2*x3])

def dz(x):
    x1, x2, x3 = np.asarray(x, dtype=float)
    return np.array([
        [0, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1],
        [2*x1, 0, 0], [0, 2*x2, 0], [0, 0, 2*x3],
        [x2, x1, 0], [x3, 0, x1], [0, x3, x2],
    ])

DESIGN_X = np.vstack([
    np.array(list(product([-1.0, 1.0], repeat=3))),
    np.vstack([np.eye(3) * ALPHA, -np.eye(3) * ALPHA]),
    np.zeros((5, 3)),
])
DESIGN_MATRIX = np.vstack([z(x) for x in DESIGN_X])
assert DESIGN_X.shape == (19, 3) and DESIGN_MATRIX.shape == (19, 10)
assert np.linalg.matrix_rank(DESIGN_MATRIX) == 10

def individual_payoff(B):
    m = B.shape[1]
    optima = []
    payoff_columns = []
    starts = [
        np.zeros(3),
        *list(np.eye(3) * (0.99 * ALPHA)),
        *list(-np.eye(3) * (0.99 * ALPHA)),
    ]
    for objective in range(m):
        candidates = []
        for x0 in starts:
            result = minimize(
                lambda x, j=objective: float(z(x) @ B[:, j]),
                x0,
                jac=lambda x, j=objective: dz(x).T @ B[:, j],
                method='SLSQP',
                bounds=[(-ALPHA, ALPHA)] * 3,
                constraints={'type': 'ineq', 'fun': lambda x: ALPHA**2 - x @ x},
                options={'ftol': 1e-11, 'maxiter': 500},
            )
            candidates.append(result)
        valid = [result for result in candidates if result.success and result.x @ result.x <= ALPHA**2 + 1e-7]
        if not valid:
            raise RuntimeError(f'Nenhum ótimo individual válido para o objetivo {objective}.')
        best = min(valid, key=lambda result: float(result.fun))
        optima.append(best.x)
        payoff_columns.append(z(best.x) @ B)
    return np.asarray(optima), np.column_stack(payoff_columns)

def reconstruct_scaled_payoff(anchors, seed):
    true_responses = np.sum(
        (DESIGN_X[:, None, :] - anchors[None, :, :]) ** 2, axis=2
    )
    noise_sd = np.sqrt(true_responses.var(axis=0, ddof=1) * (0.05 / 0.95))
    rng = np.random.default_rng(int(seed))
    observations = true_responses + rng.normal(0, noise_sd, true_responses.shape)
    B = np.linalg.lstsq(DESIGN_MATRIX, observations, rcond=None)[0]
    _, payoff = individual_payoff(B)
    ideal = payoff.min(axis=1)
    amplitude = np.where(payoff.max(axis=1) - ideal > 1e-12, payoff.max(axis=1) - ideal, 1.0)
    scaled = (payoff - ideal[:, None]) / amplitude[:, None]
    return scaled

def classify_combination(sigma_pass, qc_pass):
    if sigma_pass and qc_pass:
        return 'approved'
    if (not sigma_pass) and qc_pass:
        return 'rejected_sigma_min'
    if sigma_pass and (not qc_pass):
        return 'rejected_qc'
    return 'rejected_both'

In [ ]:
parallel = pd.read_csv(PARALLEL_PATH)
runs = pd.read_csv(RUNS_PATH)
cnbi_runs = runs[(runs['method'] == 'CNBI') & runs['scenario'].isin(SCENARIOS)].copy()
assert set(parallel['scenario']) == set(SCENARIOS)
assert len(cnbi_runs) == len(SCENARIOS) * 10
assert cnbi_runs[['scenario', 'seed']].duplicated().sum() == 0

audit_rows = []
for scenario in SCENARIOS:
    m = int(scenario.split('_')[0][1:])
    scenario_path = SCENARIO_DIR / f'{scenario}_scenario.npz'
    with np.load(scenario_path, allow_pickle=False) as scenario_data:
        anchors = np.asarray(scenario_data['anchors'], dtype=float)
    assert anchors.shape == (m, 3)

    for seed in sorted(parallel.loc[parallel['scenario'] == scenario, 'seed'].unique()):
        spectral_rows = parallel[(parallel['scenario'] == scenario) & (parallel['seed'] == seed)].sort_values('singular_index')
        assert len(spectral_rows) == m - 1
        assert spectral_rows['floor'].nunique() == spectral_rows['ceiling'].nunique() == 1
        reported_floor = float(spectral_rows['floor'].iloc[0])
        reported_ceiling = float(spectral_rows['ceiling'].iloc[0])
        scaled = reconstruct_scaled_payoff(anchors, int(seed))

        global_geometry = scaled.T
        observed_spectrum = np.linalg.svd(global_geometry[1:] - global_geometry[:1], compute_uv=False)
        assert np.allclose(
            observed_spectrum, spectral_rows['singular_value'].to_numpy(),
            rtol=1e-9, atol=1e-10,
        )
        retained_d = int(spectral_rows['retained_d'].iloc[0])
        floor = float(observed_spectrum[retained_d]) if retained_d < len(observed_spectrum) else 0.0
        ceiling = float(observed_spectrum[0] / observed_spectrum[retained_d - 1])
        assert np.isclose(floor, reported_floor, rtol=1e-12, atol=1e-14)
        assert np.isclose(ceiling, reported_ceiling, rtol=1e-12, atol=1e-14)

        approved_reconstructed = set()
        for k in K_VALUES:
            for combo in combinations(range(m), k):
                combo_array = np.asarray(combo, dtype=int)
                geometry = scaled[np.ix_(combo_array, combo_array)].T
                singular_values = np.linalg.svd(geometry[1:] - geometry[:1], compute_uv=False)
                sigma_min = float(singular_values[-1])
                q_c = np.inf if sigma_min <= 1e-12 else float(singular_values[0] / sigma_min)
                sigma_pass = bool(sigma_min > floor)
                qc_pass = bool(q_c <= ceiling)
                category = classify_combination(sigma_pass, qc_pass)
                if category == 'approved':
                    approved_reconstructed.add((k, tuple(combo)))
                audit_rows.append({
                    'scenario': scenario,
                    'seed': int(seed),
                    'm': m,
                    'k': k,
                    'combination': json.dumps(list(combo)),
                    'sigma_min': sigma_min,
                    'sigma_floor': floor,
                    'sigma_pass': sigma_pass,
                    'q_c': q_c,
                    'q_c_ceiling': ceiling,
                    'q_c_pass': qc_pass,
                    'category': category,
                })

        run_row = cnbi_runs[(cnbi_runs['scenario'] == scenario) & (cnbi_runs['seed'].astype(int) == int(seed))].iloc[0]
        checkpoint_path = ROOT / run_row['checkpoint']
        with np.load(checkpoint_path, allow_pickle=False) as checkpoint:
            if 'combo_json' in checkpoint.files:
                approved_checkpoint = {
                    (int(k), tuple(json.loads(str(combo))))
                    for k, combo in zip(checkpoint['k'], checkpoint['combo_json'])
                }
            else:
                approved_checkpoint = set()
        assert approved_reconstructed == approved_checkpoint, (scenario, seed)

audit = pd.DataFrame(audit_rows)
expected_rows = 10 * len(CORRELATION_LEVELS) * sum(comb(m, k) for m in M_VALUES for k in K_VALUES)
assert len(audit) == expected_rows == 25_260
assert set(audit['category']) <= set(CATEGORY_ORDER)
audit_path = OUT_DIR / 'full_combination_filter_audit.csv'
audit.to_csv(audit_path, index=False)
print('Combinações auditadas:', len(audit))
print('Checkpoints CNBI validados:', len(cnbi_runs))

In [ ]:
stacked_summary = (
    audit.groupby(['scenario', 'category']).size()
         .unstack(fill_value=0)
         .reindex(index=SCENARIOS, columns=CATEGORY_ORDER, fill_value=0)
)
stacked_summary['total'] = stacked_summary[CATEGORY_ORDER].sum(axis=1)

cardinality = (
    audit.assign(approved=audit['category'].eq('approved'))
         .groupby(['scenario', 'k'])['approved']
         .agg(approved_count='sum', total_count='size')
         .reset_index()
)
cardinality['approval_rate'] = cardinality['approved_count'] / cardinality['total_count']
rate_matrix = (
    cardinality.pivot(index='scenario', columns='k', values='approval_rate')
               .reindex(index=SCENARIOS, columns=K_VALUES)
)
assert rate_matrix.notna().all().all()

stacked_path = OUT_DIR / 'full_filter_stacked_summary.csv'
cardinality_path = OUT_DIR / 'full_filter_approval_by_cardinality.csv'
stacked_summary.reset_index().to_csv(stacked_path, index=False)
cardinality.to_csv(cardinality_path, index=False)
print(stacked_summary.to_string())
print((100 * rate_matrix).round(1).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(FIGURE_WIDTH_CM * CM_TO_INCH, 10.5 * CM_TO_INCH))
fig.subplots_adjust(left=0.23, right=0.985, top=0.97, bottom=0.23)

proportions = stacked_summary[CATEGORY_ORDER].div(stacked_summary['total'], axis=0) * 100
y = np.array([0, 1, 2, 3.45, 4.45, 5.45, 6.90, 7.90, 8.90])
left = np.zeros(len(SCENARIOS), dtype=float)

for category in CATEGORY_ORDER:
    values = proportions[category].to_numpy(dtype=float)
    bars = ax.barh(
        y, values, left=left, height=0.72,
        color=CATEGORY_COLORS[category], edgecolor='white', linewidth=0.45,
    )
    for bar, value, base in zip(bars, values, left):
        if value >= 6.0:
            label = f'{value:.1f}%'.replace('.', ',')
            ax.text(
                base + value/2, bar.get_y() + bar.get_height()/2, label,
                ha='center', va='center', fontsize=7.2,
                color=CATEGORY_TEXT_COLORS[category],
            )
    left += values

ax.set_yticks(y, SCENARIOS)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_xlabel('Proporção no pool das 10 sementes (%)')
ax.grid(axis='x', alpha=0.20, linewidth=0.55)
ax.set_axisbelow(True)
for separator in (2.725, 6.175):
    ax.axhline(separator, color='0.35', linewidth=1.0, clip_on=False)
for spine in ax.spines.values():
    spine.set_color('0.35'); spine.set_linewidth(0.7)

legend_handles = [
    Patch(facecolor=CATEGORY_COLORS[category], label=CATEGORY_LABELS[category])
    for category in CATEGORY_ORDER
]
fig.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, 0.035), ncol=2, frameon=False)

bars_png = OUT_DIR / 'barras_empilhadas_motivos_filtro_espectral.png'
bars_pdf = OUT_DIR / 'barras_empilhadas_motivos_filtro_espectral.pdf'
fig.savefig(bars_png, dpi=300)
fig.savefig(bars_pdf, dpi=300)
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(FIGURE_WIDTH_CM * CM_TO_INCH, 10.5 * CM_TO_INCH))
fig.subplots_adjust(left=0.22, right=0.98, top=0.97, bottom=0.10)
values = 100 * rate_matrix.to_numpy(dtype=float)
image = ax.imshow(values, cmap=RDBU_NO_WHITE, vmin=0, vmax=100, aspect='auto', interpolation='nearest')

ax.set_xticks(np.arange(len(K_VALUES)), [rf'$k={k}$' for k in K_VALUES])
ax.set_yticks(np.arange(len(SCENARIOS)), SCENARIOS)
ax.tick_params(axis='both', length=0)
ax.set_xticks(np.arange(-0.5, len(K_VALUES), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(SCENARIOS), 1), minor=True)
ax.grid(which='minor', color='white', linewidth=1.2)
ax.tick_params(which='minor', bottom=False, left=False)

for row in range(values.shape[0]):
    for column in range(values.shape[1]):
        value = values[row, column]
        red, green, blue, _ = image.cmap(image.norm(value))
        luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
        text_color = 'white' if luminance < 0.52 else '0.12'
        ax.text(column, row, f'{value:.1f}%', ha='center', va='center', color=text_color, fontsize=9.0)

for separator in (2.5, 5.5):
    ax.axhline(separator, color='0.25', linewidth=1.8)
for spine in ax.spines.values():
    spine.set_color('0.35'); spine.set_linewidth(0.7)

heatmap_png = OUT_DIR / 'heatmap_taxa_aprovacao_por_cardinalidade.png'
heatmap_pdf = OUT_DIR / 'heatmap_taxa_aprovacao_por_cardinalidade.pdf'
fig.savefig(heatmap_png, dpi=300)
fig.savefig(heatmap_pdf, dpi=300)
plt.close(fig)

metadata = {
    'source_parallel_analysis': PARALLEL_PATH.relative_to(ROOT).as_posix(),
    'source_cnbi_runs': RUNS_PATH.relative_to(ROOT).as_posix(),
    'pool': '10 FULL seeds per scenario',
    'candidate_cardinalities': list(K_VALUES),
    'total_candidates': int(len(audit)),
    'sigma_rule': 'sigma_min > floor',
    'conditioning_rule': 'q_C <= ceiling',
    'categories': CATEGORY_LABELS,
    'bar_figure': '100% horizontal stacked bars pooled over 10 seeds, grouped visually by m',
    'heatmap': 'approval percentage pooled over 10 seeds for each scenario and k; two-branch RdBu uses only red/light red below 50% and light blue/blue above 50%',
    'checkpoint_validation': 'all reconstructed approved combination sets matched the 90 CNBI checkpoints',
    'publication_width_cm': FIGURE_WIDTH_CM,
}
metadata_path = OUT_DIR / 'spectral_filtering_figures_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

artifacts = [
    audit_path, stacked_path, cardinality_path,
    bars_png, bars_pdf, heatmap_png, heatmap_pdf, metadata_path,
]
for artifact in artifacts:
    assert artifact.exists() and artifact.stat().st_size > 0
print('Arquivos gerados:')
for artifact in artifacts:
    print(' -', artifact.relative_to(ROOT).as_posix())

## Leitura dos gráficos

- O gráfico de barras é 100% empilhado e mostra a composição percentual do pool das dez sementes. As nove barras horizontais seguem a ordem dos cenários e são separadas visualmente nos blocos $m=4$, $m=6$ e $m=12$. Para preservar a legibilidade, somente segmentos com pelo menos 6% recebem rótulo interno; todas as contagens permanecem disponíveis no arquivo CSV de resumo.
- Azul representa aprovação. Três tons de vermelho distinguem falha exclusiva no piso $\sigma_{\min}$, falha exclusiva no teto $q_C$ e falha simultânea.
- O heatmap mostra a taxa de aprovação separadamente para $k=2,3,4$. A escala possui somente dois ramos cromáticos: vermelho a vermelho-claro abaixo de 50% e azul-claro a azul acima de 50%; a barra de cor foi omitida porque cada célula já apresenta seu percentual.
- As contagens representam a etapa geométrica anterior à resolução dos subproblemas C-NBI; uma combinação aprovada pelo filtro ainda pode conter pesos sem interseção factível.